In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    IntegerType, DecimalType, TimestampType, DateType, StringType
)

# Helper function to save Silver tables consistently
def save_silver_table(df, table_name):

    print(f"Writing Silver table: {table_name}")
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(table_name))
    
    row_count = spark.table(table_name).count()
    print(f"Silver table created: {table_name} ({row_count:,} rows)")
    return row_count

print("Setup complete. Ready to build Silver layer.")

In [0]:

# Read from Bronze
bronze_customers = spark.table("workspace.olist_bronze.customers")
print(f"Bronze customers rows: {bronze_customers.count():,}")

# Select, clean, cast
silver_customers = (
    bronze_customers
    .select(
        F.col("customer_id"),
        F.col("customer_unique_id"),
        F.col("customer_zip_code_prefix").cast(IntegerType()).alias("customer_zip_code_prefix"),
        F.lower(F.trim(F.col("customer_city"))).alias("customer_city"),
        F.upper(F.trim(F.col("customer_state"))).alias("customer_state"),
        F.col("ingestion_timestamp")
    )
    # Deduplicate on customer_id
    .dropDuplicates(["customer_id"])
)

# Save
save_silver_table(silver_customers, "workspace.olist_silver.customers")

# Preview
display(spark.table("workspace.olist_silver.customers").limit(10))

In [0]:
%skip

print("Validation checks:")

silver = spark.table("workspace.olist_silver.customers")

total_rows = silver.count()
unique_persons = silver.select("customer_unique_id").distinct().count()
unique_states = silver.select("customer_state").distinct().count()
null_zip = silver.filter(F.col("customer_zip_code_prefix").isNull()).count()

print(f"   Total rows (one per order): {total_rows:,}")
print(f"   Unique persons (customer_unique_id): {unique_persons:,}")
print(f"   Unique states: {unique_states}")
print(f"   Null zip codes: {null_zip:,}")
print(f"\n   Insight: {total_rows - unique_persons:,} orders came from repeat customers")

In [0]:
# Build silver.orders(for oders which are delivered) and silver.orders_all(all the entries)


# Read from Bronze
bronze_orders = spark.table("workspace.olist_bronze.orders")
print(f"Bronze orders rows: {bronze_orders.count():,}")

# Cast types and derive new columns
silver_orders_full = (
    bronze_orders
    .select(
        F.col("order_id"),
        F.col("customer_id"),
        F.lower(F.trim(F.col("order_status"))).alias("order_status"),
        F.col("order_purchase_timestamp").cast(TimestampType()).alias("order_purchase_timestamp"),
        F.col("order_approved_at").cast(TimestampType()).alias("order_approved_at"),
        F.col("order_delivered_carrier_date").cast(TimestampType()).alias("order_delivered_carrier_date"),
        F.col("order_delivered_customer_date").cast(TimestampType()).alias("order_delivered_customer_date"),
        F.col("order_estimated_delivery_date").cast(TimestampType()).alias("order_estimated_delivery_date"),
        F.col("ingestion_timestamp")
    )
    # Derive purchase_year_month (e.g., "2017-10")
    .withColumn(
        "purchase_year_month",
        F.date_format(F.col("order_purchase_timestamp"), "yyyy-MM")
    )
    # Derive delivery_days (actual delivery time)
    .withColumn(
        "delivery_days",
        F.datediff(
            F.col("order_delivered_customer_date"),
            F.col("order_purchase_timestamp")
        )
    )
    # Derive delivery_delay_days (negative = early, positive = late)
    .withColumn(
        "delivery_delay_days",
        F.datediff(
            F.col("order_delivered_customer_date"),
            F.col("order_estimated_delivery_date")
        )
    )
    # Deduplicate
    .dropDuplicates(["order_id"])
)

# Save the FULL version (all statuses) for audit
save_silver_table(silver_orders_full, "workspace.olist_silver.orders_all")

# Filter to delivered only and save as the analytics version
silver_orders_delivered = silver_orders_full.filter(F.col("order_status") == "delivered")
save_silver_table(silver_orders_delivered, "workspace.olist_silver.orders")

# Preview
print("\nPreview of silver.orders (delivered only):")
display(spark.table("workspace.olist_silver.orders").limit(10))

In [0]:
%skip

# Cell 5: Validate silver.orders

print("Validation checks:")

orders_all = spark.table("workspace.olist_silver.orders_all")
orders = spark.table("workspace.olist_silver.orders")

total_all = orders_all.count()
total_delivered = orders.count()
status_breakdown = orders_all.groupBy("order_status").count().orderBy(F.desc("count"))

print(f"\n   silver.orders_all (all statuses): {total_all:,} rows")
print(f"   silver.orders (delivered only): {total_delivered:,} rows")
print(f"   Filtered out: {total_all - total_delivered:,} non-delivered orders")

print("\n   Status breakdown:")
status_breakdown.show(truncate=False)

# Date range
date_range = orders.agg(
    F.min("order_purchase_timestamp").alias("earliest_order"),
    F.max("order_purchase_timestamp").alias("latest_order")
).collect()[0]

print(f"\n   Date range of delivered orders:")
print(f"     Earliest: {date_range['earliest_order']}")
print(f"     Latest:   {date_range['latest_order']}")

# Delivery stats
delivery_stats = orders.agg(
    F.avg("delivery_days").alias("avg_delivery_days"),
    F.avg("delivery_delay_days").alias("avg_delay_days"),
    F.expr("percentile_approx(delivery_days, 0.5)").alias("median_delivery_days")
).collect()[0]

print(f"\n   Delivery performance:")
print(f"     Avg delivery time: {delivery_stats['avg_delivery_days']:.1f} days")
print(f"     Median delivery time: {delivery_stats['median_delivery_days']} days")
print(f"     Avg vs estimate: {delivery_stats['avg_delay_days']:.1f} days (negative = delivered early)")

In [0]:

# Cell 6: Build silver.order_items

# Step 1: Read from Bronze
bronze_order_items = spark.table("workspace.olist_bronze.order_items")
print(f"Bronze order_items rows: {bronze_order_items.count():,}")

# Steps 2-5: Cast types and derive total_item_value
silver_order_items = (
    bronze_order_items
    .select(
        F.col("order_id"),
        F.col("order_item_id").cast(IntegerType()).alias("order_item_id"),
        F.col("product_id"),
        F.col("seller_id"),
        F.col("shipping_limit_date").cast(TimestampType()).alias("shipping_limit_date"),
        F.col("price").cast(DecimalType(10, 2)).alias("price"),
        F.col("freight_value").cast(DecimalType(10, 2)).alias("freight_value"),
        F.col("ingestion_timestamp")
    )
    # Step 5: Derive total_item_value
    .withColumn(
        "total_item_value",
        F.col("price") + F.col("freight_value")
    )
    # Step 6: Deduplicate on composite key
    .dropDuplicates(["order_id", "order_item_id"])
)

# Step 7: Save
save_silver_table(silver_order_items, "workspace.olist_silver.order_items")

# Preview
print("\nPreview of silver.order_items:")
display(spark.table("workspace.olist_silver.order_items").limit(10))

In [0]:

# Cell 7: Validate silver.order_items

print("Validation checks:")

items = spark.table("workspace.olist_silver.order_items")

total_rows = items.count()
unique_orders = items.select("order_id").distinct().count()
unique_products = items.select("product_id").distinct().count()
unique_sellers = items.select("seller_id").distinct().count()

print(f"\n   Total line items: {total_rows:,}")
print(f"   Unique orders represented: {unique_orders:,}")
print(f"   Unique products sold: {unique_products:,}")
print(f"   Unique sellers: {unique_sellers:,}")
print(f"\n   Avg items per order: {total_rows / unique_orders:.2f}")

# Money sanity check
money_stats = items.agg(
    F.sum("price").alias("total_revenue_price"),
    F.sum("freight_value").alias("total_revenue_freight"),
    F.sum("total_item_value").alias("total_gmv"),
    F.avg("price").alias("avg_item_price"),
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price")
).collect()[0]

print(f"\n   Total GMV (Gross Merchandise Value): R$ {money_stats['total_gmv']:,.2f}")
print(f"   Of which product price: R$ {money_stats['total_revenue_price']:,.2f}")
print(f"   Of which freight: R$ {money_stats['total_revenue_freight']:,.2f}")
print(f"\n   Avg item price: R$ {money_stats['avg_item_price']:.2f}")
print(f"   Min item price: R$ {money_stats['min_price']:.2f}")
print(f"   Max item price: R$ {money_stats['max_price']:,.2f}")

# Items per order distribution
print("\n   Items-per-order distribution:")
(items
    .groupBy("order_id")
    .agg(F.count("*").alias("items_in_order"))
    .groupBy("items_in_order")
    .count()
    .orderBy("items_in_order")
    .show(10, truncate=False))

In [0]:

# Cell 8: Build silver.payments_detail and silver.payments

# Step 1: Read from Bronze
bronze_payments = spark.table("workspace.olist_bronze.order_payments")
print(f"Bronze order_payments rows: {bronze_payments.count():,}")

# Phase 1: Build payments_detail (typed clean version)

silver_payments_detail = (
    bronze_payments
    .select(
        F.col("order_id"),
        F.col("payment_sequential").cast(IntegerType()).alias("payment_sequential"),
        F.lower(F.trim(F.col("payment_type"))).alias("payment_type"),
        F.col("payment_installments").cast(IntegerType()).alias("payment_installments"),
        F.col("payment_value").cast(DecimalType(10, 2)).alias("payment_value"),
        F.col("ingestion_timestamp")
    )
    .dropDuplicates(["order_id", "payment_sequential"])
)

save_silver_table(silver_payments_detail, "workspace.olist_silver.payments_detail")

# Phase 2: Build payments (one row per order)

# Step 7: Find the PRIMARY payment type per order
# (the payment method with the highest payment_value)
window_primary = Window.partitionBy("order_id").orderBy(F.desc("payment_value"))

primary_payment_type_df = (
    silver_payments_detail
    .withColumn("rank", F.row_number().over(window_primary))
    .filter(F.col("rank") == 1)
    .select(
        F.col("order_id"),
        F.col("payment_type").alias("primary_payment_type")
    )
)

# Step 8: Aggregate per order
order_aggregates = (
    silver_payments_detail
    .groupBy("order_id")
    .agg(
        F.count("*").alias("payment_method_count"),
        F.sum("payment_value").alias("total_payment_value"),
        F.max("payment_installments").alias("max_installments")
    )
)

# Step 9: Join aggregates with primary payment type
silver_payments = (
    order_aggregates
    .join(primary_payment_type_df, on="order_id", how="left")
    # Step 10: Add the split-payment flag
    .withColumn(
        "is_split_payment",
        F.when(F.col("payment_method_count") > 1, 1).otherwise(0)
    )
    .select(
        "order_id",
        "payment_method_count",
        "primary_payment_type",
        "total_payment_value",
        "max_installments",
        "is_split_payment"
    )
)

save_silver_table(silver_payments, "workspace.olist_silver.payments")

# Preview
print("\nPreview of silver.payments (one row per order):")
display(spark.table("workspace.olist_silver.payments").limit(10))

In [0]:
# ============================================================
# Cell 10 (FIXED): Build silver.reviews
# ============================================================
print("=" * 70)
print("Building silver.reviews")
print("=" * 70)

# Step 1: Read from Bronze
bronze_reviews = spark.table("workspace.olist_bronze.order_reviews")
print(f"Bronze order_reviews rows: {bronze_reviews.count():,}")

# Steps 2-6: Cast (with try_cast for review_score), clean, derive
reviews_cleaned = (
    bronze_reviews
    .select(
        F.col("review_id"),
        F.col("order_id"),
        # try_cast returns NULL if cast fails, instead of erroring
        F.expr("try_cast(review_score AS INT)").alias("review_score"),
        F.trim(F.col("review_comment_title")).alias("review_comment_title"),
        F.trim(F.col("review_comment_message")).alias("review_comment_message"),
        F.expr("try_cast(review_creation_date AS TIMESTAMP)").alias("review_creation_date"),
        F.expr("try_cast(review_answer_timestamp AS TIMESTAMP)").alias("review_answer_timestamp"),
        F.col("ingestion_timestamp")
    )
    # Drop rows where review_score couldn't be parsed 
    .filter(F.col("review_score").isNotNull())
    # keep only valid score range 
    .filter((F.col("review_score") >= 1) & (F.col("review_score") <= 5))
    # Step 5: Derive response time in hours
    .withColumn(
        "response_time_hours",
        ((F.unix_timestamp("review_answer_timestamp") - F.unix_timestamp("review_creation_date")) / 3600).cast(IntegerType())
    )
    # Step 6: Derive has_comment flag
    .withColumn(
        "has_comment",
        F.when(
            F.col("review_comment_title").isNotNull() | F.col("review_comment_message").isNotNull(),
            1
        ).otherwise(0)
    )
)

# Steps 7-8: Window function to keep ONLY the latest review per order
window_latest = Window.partitionBy("order_id").orderBy(F.desc("review_creation_date"))

silver_reviews = (
    reviews_cleaned
    .withColumn("rank", F.row_number().over(window_latest))
    .filter(F.col("rank") == 1)
    .drop("rank")
)

# Step 9: Save
save_silver_table(silver_reviews, "workspace.olist_silver.reviews")

# Preview
print("\nPreview of silver.reviews:")
display(spark.table("workspace.olist_silver.reviews").limit(10))

In [0]:

# Validate silver.reviews

print("Validation checks:")

bronze = spark.table("workspace.olist_bronze.order_reviews")
reviews = spark.table("workspace.olist_silver.reviews")

bronze_count = bronze.count()
total_reviews = reviews.count()
unique_orders = reviews.select("order_id").distinct().count()

# NEW: Estimate how many rows had bad data
bronze_with_bad_score = bronze.filter(
    F.expr("try_cast(review_score AS INT) IS NULL")
).count()

print(f"\n   Bronze rows:                          {bronze_count:,}")
print(f"   Bronze rows with bad score (filtered):{bronze_with_bad_score:,}")
print(f"   Silver reviews rows:                  {total_reviews:,}")
print(f"   Unique orders in Silver:              {unique_orders:,}")
print(f"\n   One-review-per-order check: {total_reviews == unique_orders}")

# ... rest of validation unchanged (score distribution, comments, response time)

In [0]:
#  Build silver.products


# Step 1: Read both Bronze tables
bronze_products = spark.table("workspace.olist_bronze.products")
bronze_translation = spark.table("workspace.olist_bronze.product_category_name")

print(f"Bronze products rows:            {bronze_products.count():,}")
print(f"Bronze category translation rows:{bronze_translation.count():,}")

# Steps 2-3: Cast types and derive volume
products_typed = (
    bronze_products
    .select(
        F.col("product_id"),
        F.col("product_category_name"),
        F.expr("try_cast(product_name_lenght AS INT)").alias("product_name_lenght"),
        F.expr("try_cast(product_description_lenght AS INT)").alias("product_description_lenght"),
        F.expr("try_cast(product_photos_qty AS INT)").alias("product_photos_qty"),
        F.expr("try_cast(product_weight_g AS INT)").alias("product_weight_g"),
        F.expr("try_cast(product_length_cm AS INT)").alias("product_length_cm"),
        F.expr("try_cast(product_height_cm AS INT)").alias("product_height_cm"),
        F.expr("try_cast(product_width_cm AS INT)").alias("product_width_cm"),
        F.col("ingestion_timestamp")
    )
    # Step 3: Derive product_volume_cm3
    .withColumn(
        "product_volume_cm3",
        F.col("product_length_cm") * F.col("product_height_cm") * F.col("product_width_cm")
    )
)

# Prepare the translation table
translation_clean = (
    bronze_translation
    .select(
        F.col("product_category_name"),
        F.col("product_category_name_english")
    )
    .dropDuplicates(["product_category_name"])
)

# Step 4: Join products with translation
products_joined = products_typed.join(
    translation_clean,
    on="product_category_name",
    how="left"
)

# Step 5: Fill nulls in both category columns with "unknown"
silver_products = (
    products_joined
    .withColumn(
        "product_category_name",
        F.coalesce(F.col("product_category_name"), F.lit("unknown"))
    )
    .withColumn(
        "product_category_name_english",
        F.coalesce(F.col("product_category_name_english"), F.lit("unknown"))
    )
    # Step 6: Deduplicate
    .dropDuplicates(["product_id"])
    # Reorder columns for readability
    .select(
        "product_id",
        "product_category_name",
        "product_category_name_english",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
        "product_volume_cm3",
        "ingestion_timestamp"
    )
)

# Step 7: Save
save_silver_table(silver_products, "workspace.olist_silver.products")

# Preview
print("\nPreview of silver.products:")
display(spark.table("workspace.olist_silver.products").limit(10))

In [0]:

# Step 1: Read from Bronze
bronze_sellers = spark.table("workspace.olist_bronze.sellers")
print(f"Bronze sellers rows: {bronze_sellers.count():,}")

# Steps 2-4: Cast and clean
silver_sellers = (
    bronze_sellers
    .select(
        F.col("seller_id"),
        F.expr("try_cast(seller_zip_code_prefix AS INT)").alias("seller_zip_code_prefix"),
        F.lower(F.trim(F.col("seller_city"))).alias("seller_city"),
        F.upper(F.trim(F.col("seller_state"))).alias("seller_state"),
        F.col("ingestion_timestamp")
    )
    # Step 5: Deduplicate
    .dropDuplicates(["seller_id"])
)

# Step 6: Save
save_silver_table(silver_sellers, "workspace.olist_silver.sellers")

# Preview
print("\nPreview of silver.sellers:")
display(spark.table("workspace.olist_silver.sellers").limit(10))

In [0]:

print("Validation checks:")

sellers = spark.table("workspace.olist_silver.sellers")
total = sellers.count()
unique = sellers.select("seller_id").distinct().count()
states = sellers.select("seller_state").distinct().count()
cities = sellers.select("seller_city").distinct().count()

print(f"\n   Total sellers:        {total:,}")
print(f"   Unique seller IDs:    {unique:,}")
print(f"   Unique check:         {total == unique}")
print(f"   Unique states:        {states}")
print(f"   Unique cities:        {cities:,}")

# Geographic distribution
print("\n   Seller distribution by state (top 10):")
(sellers
    .groupBy("seller_state")
    .count()
    .orderBy(F.desc("count"))
    .show(10, truncate=False))

# Top seller cities
print("\n   Top seller cities (top 10):")
(sellers
    .groupBy("seller_city", "seller_state")
    .count()
    .orderBy(F.desc("count"))
    .show(10, truncate=False))

# Null zip check
null_zip = sellers.filter(F.col("seller_zip_code_prefix").isNull()).count()
print(f"\n   Sellers with null zip: {null_zip:,}")

In [0]:

# Build silver.geolocation


# Step 1: Read from Bronze
bronze_geo = spark.table("workspace.olist_bronze.geolocation")
print(f"Bronze geolocation rows: {bronze_geo.count():,}")

# Steps 2-3: Cast and clean
geo_cleaned = (
    bronze_geo
    .select(
        F.expr("try_cast(geolocation_zip_code_prefix AS INT)").alias("geolocation_zip_code_prefix"),
        F.expr("try_cast(geolocation_lat AS DECIMAL(10,7))").alias("geolocation_lat"),
        F.expr("try_cast(geolocation_lng AS DECIMAL(10,7))").alias("geolocation_lng"),
        F.lower(F.trim(F.col("geolocation_city"))).alias("geolocation_city"),
        F.upper(F.trim(F.col("geolocation_state"))).alias("geolocation_state"),
        F.col("ingestion_timestamp")
    )
    
)

# Step 4-5: Aggregate per zip prefix
# We need:
#   - avg lat, avg lng (simple)
#   - count (simple)
#   - most common city, most common state (requires a sub-aggregation)

# Sub-step: Find the most common (city, state) per zip
# Approach: count (zip, city, state) combinations, then keep top-1 per zip
city_counts = (
    geo_cleaned
    .groupBy("geolocation_zip_code_prefix", "geolocation_city", "geolocation_state")
    .agg(F.count("*").alias("combo_count"))
)

# Rank combos within each zip, keep rank 1 (most common)
window_mode = Window.partitionBy("geolocation_zip_code_prefix").orderBy(F.desc("combo_count"))

most_common_city_state = (
    city_counts
    .withColumn("rank", F.row_number().over(window_mode))
    .filter(F.col("rank") == 1)
    .select(
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "geolocation_state"
    )
)

# Main aggregation: average coordinates + count
coords_avg = (
    geo_cleaned
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        F.avg("geolocation_lat").cast(DecimalType(10, 7)).alias("geolocation_lat"),
        F.avg("geolocation_lng").cast(DecimalType(10, 7)).alias("geolocation_lng"),
        F.count("*").alias("sample_count"),
        F.max("ingestion_timestamp").alias("ingestion_timestamp")
    )
)

# Step 6: Join the two pieces together
silver_geolocation = coords_avg.join(
    most_common_city_state,
    on="geolocation_zip_code_prefix",
    how="left"
).select(
    "geolocation_zip_code_prefix",
    "geolocation_lat",
    "geolocation_lng",
    "geolocation_city",
    "geolocation_state",
    "sample_count",
    "ingestion_timestamp"
)

# Step 7: Save
save_silver_table(silver_geolocation, "workspace.olist_silver.geolocation")

# Preview
print("\nPreview of silver.geolocation:")
display(spark.table("workspace.olist_silver.geolocation").orderBy(F.desc("sample_count")).limit(10))

In [0]:

# Build silver.category_translation

# Step 1: Read from Bronze
bronze_translation = spark.table("workspace.olist_bronze.product_category_name")
print(f"Bronze category translation rows: {bronze_translation.count():,}")

# Steps 2-3: Clean and deduplicate
silver_category_translation = (
    bronze_translation
    .select(
        F.lower(F.trim(F.col("product_category_name"))).alias("product_category_name"),
        F.lower(F.trim(F.col("product_category_name_english"))).alias("product_category_name_english"),
        F.col("ingestion_timestamp")
    )
    .dropDuplicates(["product_category_name"])
)

# Step 4: Save
save_silver_table(silver_category_translation, "workspace.olist_silver.category_translation")

# Preview
print("\nPreview of silver.category_translation:")
display(spark.table("workspace.olist_silver.category_translation").orderBy("product_category_name"))

In [0]:
# Cell 20: Silver Layer — Final Wrap-Up Validation

# All Silver tables
silver_tables = [
    "customers",
    "orders",
    "orders_all",
    "order_items",
    "payments",
    "payments_detail",
    "reviews",
    "products",
    "sellers",
    "geolocation",
    "category_translation"
]

print("\nRow counts across the Silver layer:")
print(f"{'Table':<25} {'Rows':>15}")
print("-" * 42)
for t in silver_tables:
    count = spark.table(f"workspace.olist_silver.{t}").count()
    print(f"{t:<25} {count:>15,}")

# Cross-table integrity checks
print("\n" + "=" * 70)
print("CROSS-TABLE INTEGRITY CHECKS")
print("=" * 70)

customers = spark.table("workspace.olist_silver.customers")
orders = spark.table("workspace.olist_silver.orders")
items = spark.table("workspace.olist_silver.order_items")
payments = spark.table("workspace.olist_silver.payments")
reviews = spark.table("workspace.olist_silver.reviews")
products = spark.table("workspace.olist_silver.products")

# Check 1: Every order in items should exist in orders
orders_in_items = items.select("order_id").distinct()
orders_in_orders = orders.select("order_id").distinct()
missing_from_orders = orders_in_items.join(orders_in_orders, "order_id", "left_anti").count()
print(f"\nOrders in items but missing from silver.orders: {missing_from_orders:,}")
print("   (Expected: ~190 — these are non-delivered orders with items)")

# Check 2: Every product in items should exist in products
products_in_items = items.select("product_id").distinct()
products_in_products = products.select("product_id").distinct()
missing_products = products_in_items.join(products_in_products, "product_id", "left_anti").count()
print(f"\nProducts in items but missing from silver.products: {missing_products:,}")
print("   (Expected: 0 — every sold product should be in the catalog)")

# Check 3: Every order in payments should be a real order
orders_with_payment = payments.select("order_id").distinct()
missing_orders_payment = orders_with_payment.join(orders_in_orders, "order_id", "left_anti").count()
print(f"\nOrders in payments but missing from silver.orders: {missing_orders_payment:,}")
print("   (Expected: ~3000 — non-delivered orders still had payments attempted)")

# Check 4: Reviews should mostly map to orders
orders_with_review = reviews.select("order_id").distinct()
missing_orders_review = orders_with_review.join(orders_in_orders, "order_id", "left_anti").count()
print(f"\nOrders in reviews but missing from silver.orders: {missing_orders_review:,}")
print("   (Expected: small number — most reviews are for delivered orders)")

print("\n" + "=" * 70)
print("SILVER LAYER COMPLETE")
print("=" * 70)